# Example - Change of Institute (Li and Pearl, 2024)

In this notebook, the results of the "Change of Institute" example are replicated.

References:

> **"Probabilities of Causation with Nonbinary Treatment and Effect"**
> 
> https://ojs.aaai.org/index.php/AAAI/article/view/30030


> **"Probabilities of Causation with Nonbinary Treatment and Effect"**
> 
> arXiv:2208.09568v1
> 
> https://arxiv.org/pdf/2208.09568

In [1]:
import sys
sys.path.insert(0, '../src/')

import numpy as np

from probabilities_of_causation import *

## Treatments and outcomes 

In [2]:
treatment = {'x1': 'No', 'x2': 'A', 'x3': 'B', 'x3': 'C'}
outcome = {'y1': 'success', 'y2': 'failure'} 

## Experimental and observational data

See paper `Table 3: Experimental data collected by Bob.`. 

|             | y1: Success | y2: Failure | Overall |
|-------------|---------|---------|---------|
| x1: No institute | 53 | 247 | 300 |
| x2: Institute A | 269 | 31 | 300 |
| x3: Institute B | 234 | 66 | 300 |
| x4: Institute C | 151 | 149 | 300 |

In [3]:
experimental_data = {
    ('y1', 'x1'): 53/300,
    ('y2', 'x1'): 247/300, 
    ('y1', 'x2'): 269/300, 
    ('y2', 'x2'): 31/300, 
    ('y1', 'x3'): 234/300,  
    ('y2', 'x3'): 66/300,   
    ('y1', 'x4'): 151/300,  
    ('y2', 'x4'): 149/300  
}

See paper `Table 4: Observational data collected by Bob`. 

|             | y1: Success | y2: Failure | Overall |
|-------------|---------|---------|---------|
| x1: No institute | 92 | 58 | 150 |
| x2: Institute A | 55 | 118 | 173 |
| x3: Institute B | 24 | 231 | 255 |
| x4: Institute C | 599 | 23 | 622 |

In [4]:

observational_data = {
    ('x1', 'y1'): 92/1200,
    ('x1', 'y2'): 58/1200,
    ('x2', 'y1'): 55/1200,
    ('x2', 'y2'): 118/1200,
    ('x3', 'y1'): 24/1200,
    ('x3', 'y2'): 231/1200,
    ('x4', 'y1'): 599/1200,
    ('x4', 'y2'): 23/1200
}

In [5]:
probabilities = Probabilities(observational_data, experimental_data)


=== Data Completeness Check ===

✓ All observational probabilities P(x, y) are present

✓ All experimental probabilities P(y|do(x)) are present

=== Marginal Probability Check ===
P(x1) = 0.1250
P(x2) = 0.1442
P(x3) = 0.2125
P(x4) = 0.5183
P(y1) = 0.6417
P(y2) = 0.3583
=== End of Data Completeness Check ===



## Queries and verification of bounds

Bob went to institute A (x2), but he failed on the job market (y2). 

Bob wants to know what would happen if he chose the other two institutes. Bob's question is expressed as the following probabilities of causation:

- $P({y_1}_{x_3}|x_2, y_2)=P({y_1}_{x_3},x_2, y_2)/P(x_2,y_2)$
- $P({y_1}_{x_4}|x_2, y_2)=P({y_1}_{x_4},x_2, y_2)/P(x_2,y_2)$




### Query  $P({y_1}_{x_3}|x_2, y_2)=P({y_1}_{x_3},x_2, y_2)/P(x_2,y_2)$


In [6]:
 query = {
    "hypotheticals": [('y1', 'x3')],
    "x_p": 'x2',
    "y_q": 'y2'
}

In [7]:
lb, ub = get_bounds(query, probabilities)
print(f"Lower bound: {lb:.3f}\nUpper bound: {ub:.3f} ")

Applying theorem 7: probability of necessity
Lower bound: 0.071
Upper bound: 0.098 


In [8]:
p_x2_y2 = probabilities.p_obs.get(('x2', 'y2'), 0)
print(f"{p_x2_y2:3f}")

0.098333


In [9]:
cond_lb = lb/p_x2_y2
cond_ub = ub/p_x2_y2
print(f"Conditional lower bound: {cond_lb:.3f}\nConditional upper bound: {cond_ub:.3f} ")

Conditional lower bound: 0.720
Conditional upper bound: 1.000 


According to the paper:

$0.720 \le P({y_1}_{x_3}|x_2, y_2) \le 1$
    

In [10]:
np.isclose(0.720, cond_lb, atol=1e-3)

np.True_

In [11]:
np.isclose(1.0, cond_ub, atol=1e-3)

np.True_

### Query $P({y_1}_{x_4}|x_2, y_2)=P({y_1}_{x_4},x_2, y_2)/P(x_2,y_2)$


In [12]:
 query = {
    "hypotheticals": [('y1', 'x4')],
    "x_p": 'x2',
    "y_q": 'y2'
}

In [13]:
lb, ub = get_bounds(query, probabilities)

print(f"Lower bound: {lb:.3f}\nUpper bound: {ub:.3f} ")

Applying theorem 7: probability of necessity
Lower bound: 0.000
Upper bound: 0.004 


In [14]:
cond_lb = lb/p_x2_y2
cond_ub = ub/p_x2_y2

print(f"Conditional lower bound: {cond_lb:.3f}\nConditional upper bound: {cond_ub:.3f} ")

Conditional lower bound: 0.000
Conditional upper bound: 0.042 


According to the paper:

$0 \le P({y_1}_{x_4}|x_2, y_2) \le 0.042$

In [15]:
np.isclose(0, cond_lb, atol=1e-3)

np.True_

In [16]:
np.isclose(0.042, cond_ub, atol=1e-3)

np.True_

##  Intermediate calculation steps reported in the paper
See `Appendix/Calculation in the Examples/Change of Institute`

> https://arxiv.org/pdf/2208.09568


### By Theorem 7:

$\large 85/1200 \leq P({y_1}_{x_3},x_2,y_2) \ge 118/1200$

In [17]:
query = {
    "hypotheticals": [('y1', 'x3')],
    "x_p": 'x2',
    "y_q": 'y2'
}

lb, ub = get_bounds(query, probabilities)

print(f"Lower bound: {lb:.3f}\nUpper bound: {ub:.3f} ")

Applying theorem 7: probability of necessity
Lower bound: 0.071
Upper bound: 0.098 


In [18]:
85/1200

0.07083333333333333

In [19]:
118/1200

0.09833333333333333

### By Theorem 7:

$\large 0 \leq P({y_1}_{x_4},x_2,y_2) \ge 5/1200$

In [20]:
query = {
    "hypotheticals": [('y1', 'x4')],
    "x_p": 'x2',
    "y_q": 'y2'
}

lb, ub = get_bounds(query, probabilities)

print(f"Lower bound: {lb:.3f}\nUpper bound: {ub:.3f} ")

Applying theorem 7: probability of necessity
Lower bound: 0.000
Upper bound: 0.004 


In [21]:
5/1200

0.004166666666666667